In [ ]:
import random
import copy

# --- 1. Grid and Component Definition ---

# Component dimensions (width, height)
COMPONENTS = {
    "ALU": (5, 5),
    "Cache": (7, 4),
    "Control Unit": (4, 4),
    "Register File": (6, 6),
    "Decoder": (5, 3),
    "Floating Unit": (5, 5),
}

# The order of components must be consistent
COMPONENT_NAMES = list(COMPONENTS.keys())
COMPONENT_SPECS = list(COMPONENTS.values())

# Assumed netlist for wiring calculation {Net: [Component_Name_1, Component_Name_2, ...]}
NETLIST = {
    "Net1": ["ALU", "Cache", "Control Unit"],
    "Net2": ["Cache", "Register File"],
    "Net3": ["Control Unit", "Decoder", "Floating Unit"],
    "Net4": ["Register File", "Floating Unit"],
}

# GA Parameters
GRID_SIZE = 25
POPULATION_SIZE = 6
MAX_GENERATIONS = 15
MUTATION_RATE = 0.08  # 10%
ELITISM_COUNT = 1     # Carry forward the best 1 individual

# Fitness function weights (penalize overlap the most)
W_OVERLAP = 1000.0
W_AREA = 1.0
W_WIRE = 1.0

# --- 2. Chromosome Representation and Initialization ---

def create_individual():
    """
    Creates a single chromosome representing a layout.
    A chromosome is a list of [x1, y1, x2, y2, ...].
    """
    chromosome = []
    for width, height in COMPONENT_SPECS:
        # Ensure the component is placed fully within the grid
        x = random.randint(0, GRID_SIZE - width)
        y = random.randint(0, GRID_SIZE - height)
        chromosome.extend([x, y])
    return chromosome

def create_initial_population(size):
    """Creates the initial population of random layouts."""
    return [create_individual() for _ in range(size)]

# --- 3. Fitness Function Implementation ---

def decode_chromosome(chromosome):
    """Converts a chromosome list into a list of component dictionaries."""
    layout = []
    for i in range(len(COMPONENT_NAMES)):
        name = COMPONENT_NAMES[i]
        width, height = COMPONENT_SPECS[i]
        x = chromosome[i*2]
        y = chromosome[i*2 + 1]
        layout.append({
            "name": name,
            "x": x, "y": y,
            "width": width, "height": height
        })
    return layout

def calculate_fitness(chromosome):
    """Calculates the fitness of a chromosome, where lower is better."""
    layout = decode_chromosome(chromosome)

    # a) Calculate Total Overlap
    total_overlap = 0
    for i in range(len(layout)):
        for j in range(i + 1, len(layout)):
            comp1 = layout[i]
            comp2 = layout[j]

            dx = max(0, min(comp1['x'] + comp1['width'], comp2['x'] + comp2['width']) - max(comp1['x'], comp2['x']))
            dy = max(0, min(comp1['y'] + comp1['height'], comp2['y'] + comp2['height']) - max(comp1['y'], comp2['y']))
            total_overlap += dx * dy

    # b) Calculate Total Bounding Area
    max_x = max(c['x'] + c['width'] for c in layout)
    max_y = max(c['y'] + c['height'] for c in layout)
    min_x = min(c['x'] for c in layout)
    min_y = min(c['y'] for c in layout)
    bounding_area = (max_x - min_x) * (max_y - min_y)

    # c) Calculate Total Wiring Distance (HPWL)
    total_wire_length = 0
    component_centers = {c['name']: (c['x'] + c['width']/2, c['y'] + c['height']/2) for c in layout}

    for net_components in NETLIST.values():
        net_points_x = [component_centers[name][0] for name in net_components]
        net_points_y = [component_centers[name][1] for name in net_components]

        hpwl_x = max(net_points_x) - min(net_points_x)
        hpwl_y = max(net_points_y) - min(net_points_y)
        total_wire_length += hpwl_x + hpwl_y

    # d) Calculate Final Weighted Fitness
    fitness = (W_OVERLAP * total_overlap) + \
              (W_AREA * bounding_area) + \
              (W_WIRE * total_wire_length)

    return fitness, total_overlap, bounding_area, total_wire_length

# --- 4. Genetic Operators ---

def select_parents(population):
    """Selects two parents randomly from the population."""
    return random.sample(population, 2)

def crossover(parent1, parent2):
    """Performs single-point crossover to create two offspring."""
    split_point = random.randint(1, len(parent1) - 2)
    offspring1 = parent1[:split_point] + parent2[split_point:]
    offspring2 = parent2[:split_point] + parent1[split_point:]
    return offspring1, offspring2

def mutate(chromosome):
    """
    Mutates a chromosome by randomly changing the coordinates of one component.
    """
    if random.random() < MUTATION_RATE:
        # Pick a random component (gene pair) to mutate
        component_idx = random.randint(0, len(COMPONENT_NAMES) - 1)
        width, height = COMPONENT_SPECS[component_idx]

        # Generate new random coordinates for this component
        new_x = random.randint(0, GRID_SIZE - width)
        new_y = random.randint(0, GRID_SIZE - height)

        chromosome[component_idx*2] = new_x
        chromosome[component_idx*2 + 1] = new_y
    return chromosome

# --- 5. Main GA Loop ---

def run_genetic_algorithm():
    """The main GA loop."""
    # Initialization
    population = create_initial_population(POPULATION_SIZE)
    best_solution = None
    best_fitness = float('inf')

    print(f"--- Starting Genetic Algorithm ---")
    print(f"Population: {POPULATION_SIZE}, Generations: {MAX_GENERATIONS}\n")

    for generation in range(MAX_GENERATIONS):
        # Evaluate fitness of the current population
        fitness_scores = [calculate_fitness(ind) for ind in population]

        # Combine individuals with their fitness scores for sorting
        pop_with_fitness = list(zip(population, fitness_scores))

        # Sort by fitness (lower is better)
        pop_with_fitness.sort(key=lambda x: x[1][0])

        # Check for new best solution
        if pop_with_fitness[0][1][0] < best_fitness:
            best_fitness = pop_with_fitness[0][1][0]
            best_solution = pop_with_fitness[0]

        print(f"Generation {generation+1}: Best Fitness = {best_fitness:.2f}")

        # Create the next generation
        new_population = []

        # Elitism: Carry over the best individuals
        elites = [ind[0] for ind in pop_with_fitness[:ELITISM_COUNT]]
        new_population.extend(elites)

        # Create the rest of the new population from offspring
        while len(new_population) < POPULATION_SIZE:
            # Selection
            parent1, parent2 = select_parents(population)

            # Crossover
            offspring1, offspring2 = crossover(parent1, parent2)

            # Mutation
            new_population.append(mutate(offspring1))
            if len(new_population) < POPULATION_SIZE:
                new_population.append(mutate(offspring2))

        population = new_population

    # --- 6. Output the best result ---
    print("\n--- Genetic Algorithm Finished ---")

    final_chromosome, final_fitness_metrics = best_solution
    final_fitness, final_overlap, final_area, final_wire_length = final_fitness_metrics

    print(f"\nBest Fitness Value: {final_fitness:.2f}")
    print(f"Corresponding Total Overlap: {final_overlap:.2f}")
    print(f"Corresponding Bounding Box Area: {final_area:.2f}")
    print(f"Corresponding Total Wiring Length: {final_wire_length:.2f}")

    print("\nOptimal Placement (Bottom-Left Coordinates):")
    decoded_layout = decode_chromosome(final_chromosome)
    for component in decoded_layout:
        print(f"- {component['name']}: (x={component['x']}, y={component['y']})")

# Run the algorithm
if __name__ == "__main__":
    run_genetic_algorithm()

--- Starting Genetic Algorithm ---
Population: 6, Generations: 15

Generation 1: Best Fitness = 4657.00
Generation 2: Best Fitness = 4657.00
Generation 3: Best Fitness = 4657.00
Generation 4: Best Fitness = 2618.50
Generation 5: Best Fitness = 2618.50
Generation 6: Best Fitness = 2556.50
Generation 7: Best Fitness = 2452.50
Generation 8: Best Fitness = 2452.50
Generation 9: Best Fitness = 577.00
Generation 10: Best Fitness = 577.00
Generation 11: Best Fitness = 577.00
Generation 12: Best Fitness = 577.00
Generation 13: Best Fitness = 577.00
Generation 14: Best Fitness = 537.50
Generation 15: Best Fitness = 493.50

--- Genetic Algorithm Finished ---

Best Fitness Value: 493.50
Corresponding Total Overlap: 0.00
Corresponding Bounding Box Area: 420.00
Corresponding Total Wiring Length: 73.50

Optimal Placement (Bottom-Left Coordinates):
- ALU: (x=18, y=12)
- Cache: (x=5, y=17)
- Control Unit: (x=7, y=7)
- Register File: (x=13, y=0)
- Decoder: (x=14, y=8)
- Floating Unit: (x=3, y=2)
